# LAFO Mean-Reversion Trading System - Exploration Notebook

This notebook demonstrates the complete LAFO trading pipeline: synthetic data generation → model training → backtesting → performance analysis.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Custom modules
from simulation import generate_piecewise_trendarma, load_real_sp500
from lafo import lafo_loss, lafo_loss_efficient, build_sliding_average_operator
from penalized_lafo import lafo_l2_closed_form, lafo_tv_solver
from cnn_filter import LAFOCNN
from trading_backtest import backtest, compute_signal, compute_position

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

## 2. Generate Synthetic Data

In [ ]:
# Generate synthetic data with regime changes
T = 1200
R = 5  # Number of regimes

prices, true_fair_value, regime_labels = generate_piecewise_trendarma(
    T=T, R=R, p=2, q=1, seed=42, df=4
)

print(f"Generated synthetic data: {len(prices)} samples")
print(f"True fair value range: [{true_fair_value.min():.2f}, {true_fair_value.max():.2f}]")
print(f"Price range: [{prices.min():.2f}, {prices.max():.2f}]")
print(f"Number of regimes: {len(np.unique(regime_labels))}")

## 3. LAFO Loss Function Test

In [ ]:
# Simple moving average as baseline filter
K = 80  # Window size
sma = np.convolve(prices, np.ones(K)/K, mode='valid')
sma = np.pad(sma, (T-len(sma), 0), 'edge')

# Compute LAFO loss
loss_matrix = lafo_loss(prices, sma, K)
loss_efficient = lafo_loss_efficient(prices, sma, K)

print(f"LAFO Loss (matrix): {loss_matrix:.6f}")
print(f"LAFO Loss (efficient): {loss_efficient:.6f}")
print(f"Difference: {abs(loss_matrix - loss_efficient):.10f}")

## 4. Train CNN Filter

In [ ]:
# Initialize CNN model
K = 512  # Kernel size as per paper
model = LAFOCNN(num_channels=48, kernel_size=K)

# Prepare data
X = torch.from_numpy(prices).float()
y = torch.from_numpy(prices).float()

# Train
model.fit(
    y=y.numpy(),
    K=K,
    num_epochs=50,
    lr=0.008
)

# Get predictions
with torch.no_grad():
    predictions = model.forward(X)
    predictions_np = predictions.cpu().numpy().squeeze()

print(f"Training completed")
print(f"Final prediction shape: {predictions_np.shape}")

## 5. Backtest Results

In [ ]:
# Run backtest
params = {
    'tau_entry': 0.0005,
    'alpha': 200,
    'c': 0.0003,
}

results = backtest(
    prices=prices,
    filtered_value=predictions_np,
    residuals=prices - predictions_np,
    **params
)

print("\n=== Backtest Results ===")
print(f"Turnover:       {results['turnover']:.4f}")
print(f"Sharpe Ratio:   {results['sharpe_ratio']:.4f}")
print(f"Hit Rate:       {results['hit_rate']:.4f}")
print(f"Excess Return:  {results['excess_return']:.4f}")
print(f"Total Return:   {results['total_return']:.4f}")
print(f"Max Drawdown:   {results['max_drawdown']:.4f}")

## 6. Visualize Results

In [ ]:
# Plot price, filter, and residuals
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

# Normalize for plotting
norm_prices = (prices - prices.min()) / (prices.max() - prices.min())
norm_predictions = (predictions_np - predictions_np.min()) / (predictions_np.max() - predictions_np.min())
norm_residuals = ((prices - predictions_np) - (prices - predictions_np).min()) / \
                 ((prices - predictions_np).max() - (prices - predictions_np).min())

axes[0].plot(norm_prices, label='Price', linewidth=1)
axes[0].plot(norm_predictions, label='CNN Filter', linestyle='--', linewidth=1)
axes[0].legend()
axes[0].set_ylabel('Normalized')
axes[0].grid(True, alpha=0.3)

axes[1].plot(norm_residuals, label='Residuals', linewidth=1)
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_ylabel('Residuals')
axes[1].grid(True, alpha=0.3)

# Rolling statistics
window = 50
rolling_mean = pd.Series(norm_residuals).rolling(window=window).mean()
axes[2].plot(rolling_mean, label='Rolling Mean', linewidth=1)
axes[2].axhline(y=0, color='k', linestyle='--', linewidth=0.5)
axes[2].set_ylabel('Rolling Mean')
axes[2].grid(True, alpha=0.3)

# Positions
positions = compute_position(norm_residuals, params['tau_entry'], params['alpha'])
axes[3].plot(positions, label='Positions', linewidth=1, color='orange')
axes[3].axhline(y=0, color='k', linestyle='--', linewidth=0.5)
axes[3].set_ylabel('Position Size')
axes[3].set_xlabel('Time')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/filter_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nResults saved to filter_performance.png")

## 7. Hyperparameter Comparison

In [ ]:
# Test different K values
K_values = [32, 64, 128, 256, 512]
results_dict = []

for K in K_values:
    # Train with different K
    model = LAFOCNN(num_channels=48, kernel_size=K)
    model.fit(y=y.numpy(), K=K, num_epochs=30, lr=0.008)
    
    with torch.no_grad():
        predictions = model.forward(X).cpu().numpy().squeeze()
        
    results = backtest(
        prices=prices,
        filtered_value=predictions,
        residuals=prices - predictions,
        tau_entry=0.001,
        alpha=100,
        c=0.0002
    )
    
    results_dict.append({
        'K': K,
        'Sharpe': results['sharpe_ratio'],
        'Turnover': results['turnover'],
        'Hit Rate': results['hit_rate']
    })

# Create DataFrame and plot
results_df = pd.DataFrame(results_dict)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(results_df['K'], results_df['Sharpe'], 'o-', linewidth=2)
axes[0].set_xlabel('K (window size)')
axes[0].set_ylabel('Sharpe Ratio')
axes[0].set_title('Sharpe Ratio vs K')
axes[0].grid(True, alpha=0.3)

axes[1].plot(results_df['K'], results_df['Turnover'], 's-', linewidth=2)
axes[1].set_xlabel('K (window size)')
axes[1].set_ylabel('Turnover')
axes[1].set_title('Turnover vs K')
axes[1].grid(True, alpha=0.3)

axes[2].plot(results_df['K'], results_df['Hit Rate'], 'd-', linewidth=2)
axes[2].set_xlabel('K (window size)')
axes[2].set_ylabel('Hit Rate')
axes[2].set_title('Hit Rate vs K')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/hyperparameter_k.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nHyperparameter results saved to hyperparameter_k.png")
print(f"\nBest K: {results_df.loc[results_df['Sharpe'].idxmax(), 'K']}")
print(f"Best Sharpe: {results_df['Sharpe'].max():.4f}")

## 8. Load Real SP500 Data (Optional)

In [ ]:
# Uncomment to load real data
# prices = load_real_sp500(start_date="2024-01-01")
# print(f"Loaded {len(prices)} days of SP500 data")
# print(f"Date range: {pd.Series(prices.index[0]).date()} to {pd.Series(prices.index[-1]).date()}")

## 9. Summary Statistics

In [ ]:
# Print summary
print("=== LAFO Mean-Reversion System Summary ===")
print(f"\nConfiguration:")
print(f"  - Synthetic data length: {len(prices)} samples")
print(f"  - Regimes: {len(np.unique(regime_labels))}")
print(f"  - CNN kernel size: {K}")
print(f"  - Training epochs: 50")

print(f"\nPerformance Metrics:")
print(f"  - Turnover: {results['turnover']:.4f}")
print(f"  - Sharpe Ratio: {results['sharpe_ratio']:.4f}")
print(f"  - Hit Rate: {results['hit_rate']:.4f}")
print(f"  - Excess Return: {results['excess_return']:.4f}")
print(f"  - Max Drawdown: {results['max_drawdown']:.4f}")

print(f"\nTrading Activity:")
print(f"  - Active trading days: {np.sum(np.abs(results['positions']) > 0)}")
print(f"  - Total trades: {len(results['positions'])}")